# Laboratorio 08.1 — Function calling y ejecución de herramientas

Separaremos generación, salida estructurada, selección de herramienta y ejecución real con contratos Pydantic y funciones Python locales.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Distinguir texto generado, JSON estructurado, solicitud de herramienta y resultado de ejecución.
- Validar argumentos de herramienta con Pydantic antes de ejecutarlos.
- Inspeccionar esquemas de calculadora, clima mock, productos y documentos.
- Medir exactitud de selección y limitar las llamadas opcionales a una.

## 2. Conceptos

Generar texto produce contenido sin contrato de ejecución. Salida estructurada fuerza una forma de datos, pero no implica una acción. Function calling permite que un modelo proponga nombre y argumentos de herramienta; la aplicación aún valida permisos y argumentos, ejecuta código y devuelve el resultado. Nunca ejecutes directamente texto arbitrario del modelo.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Solicitud de usuario] --> B[Modelo propone llamada]
    C[Catalogo y JSON Schema] --> B
    B --> D[Validacion Pydantic]
    D --> E{Herramienta permitida?}
    E -->|si| F[Funcion Python]
    E -->|no| G[Rechazo]
    F --> H[Resultado tipado]
    H --> I[Modelo o usuario]
```

## 4. Preparación del entorno

Usamos Pydantic y el registro local de herramientas. Todas las funciones del experimento son deterministas o simuladas; no hay servicios meteorológicos, catálogos remotos ni llamadas a modelos por defecto.

In [1]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import sys

raices_candidatas = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
raiz_curso = next(
    (
        candidato
        for raiz in raices_candidatas
        for candidato in (raiz / 'genai', raiz)
        if (candidato / 'src' / 'genai').is_dir()
    ),
    None,
)
if raiz_curso is None:
    raise FileNotFoundError('No se encontró el proyecto genAI con src/genai.')

sys.path.insert(0, str(raiz_curso / 'src'))

In [4]:
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field

from genai.environment import preparar_entorno
from genai.tools import ToolName, execute_tool, tool_catalog
from genai.tools.langchain_tools import crear_herramientas_langchain

raiz_curso = preparar_entorno()

## 5. Definición del problema

Responderemos consultas de soporte que podrían requerir cálculo, clima de ejemplo, búsqueda de producto o evidencia documental. La referencia del experimento es una decisión de herramienta etiquetada manualmente; no se intenta evaluar un proveedor con un solo ejemplo.

## 6. Línea base

Una respuesta en lenguaje natural puede decir ‘usaré la calculadora’ sin crear una llamada estructurada. Comparamos ese texto con un objeto explícito para ver la diferencia entre intención verbal y contrato ejecutable.

In [5]:
texto_generado = 'Voy a calcular el total para ti.'
print('Texto generado:', texto_generado)
print('Herramientas disponibles:', [herramienta['name'] for herramienta in tool_catalog()])
print('Esquema de calculadora:', tool_catalog()[0]['parameters'])

Texto generado: Voy a calcular el total para ti.
Herramientas disponibles: ['calculadora', 'clima_mock', 'busqueda_productos', 'busqueda_documentos']
Esquema de calculadora: {'additionalProperties': False, 'description': 'Argumentos permitidos para la calculadora aritmetica.', 'properties': {'expression': {'maxLength': 100, 'minLength': 1, 'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'CalculatorArgs', 'type': 'object'}


## 7. Implementación

Definimos el contrato de una solicitud de herramienta y simulamos una llamada generada. La validación confirma forma y campos permitidos; después el dispatcher local ejecuta solo un nombre registrado.

In [6]:
class SolicitudHerramienta(BaseModel):
    model_config = ConfigDict(extra='forbid')

    nombre: str = Field(min_length=1)
    argumentos: dict[str, object]


llamada_generada = SolicitudHerramienta.model_validate({
    'nombre': 'calculadora',
    'argumentos': {'expression': '18 * 4'},
})
resultado = execute_tool(llamada_generada.nombre, llamada_generada.argumentos)
print('Llamada validada:', llamada_generada.model_dump())
print('Ejecución:', resultado.model_dump())

Llamada validada: {'nombre': 'calculadora', 'argumentos': {'expression': '18 * 4'}}
Ejecución: {'tool_name': 'calculadora', 'success': True, 'output': '18 * 4 = 72', 'source_ids': [], 'error': None}


## 8. Experimentos

Comparamos decisiones simuladas con la etiqueta correcta. Cambia solo la herramienta seleccionada; cada resultado local es determinista. La invocación de un modelo queda en otra celda y limitada a una solicitud.

In [7]:
pruebas = [
    {'solicitud': 'cuanto es 18 por 4', 'esperada': 'calculadora', 'seleccionada': 'calculadora', 'argumentos': {'expression': '18 * 4'}},
    {'solicitud': 'busca mochila', 'esperada': 'busqueda_productos', 'seleccionada': 'busqueda_productos', 'argumentos': {'query': 'mochila'}},
    {'solicitud': 'donde esta mi paquete', 'esperada': 'busqueda_documentos', 'seleccionada': 'busqueda_documentos', 'argumentos': {'query': 'seguimiento paquete'}},
    {'solicitud': 'temperatura de Santiago', 'esperada': 'clima_mock', 'seleccionada': 'busqueda_documentos', 'argumentos': {'query': 'clima Santiago'}},
]
filas = []
for prueba in pruebas:
    resultado = execute_tool(prueba['seleccionada'], prueba['argumentos'])
    filas.append({
        'solicitud': prueba['solicitud'],
        'esperada': prueba['esperada'],
        'seleccionada': prueba['seleccionada'],
        'seleccion_correcta': prueba['esperada'] == prueba['seleccionada'],
        'argumentos_validos': resultado.success,
        'error': resultado.error,
    })
tabla_experimento = pd.DataFrame(filas)
display(tabla_experimento)
print('Adaptadores LangChain construidos:', len(crear_herramientas_langchain()))
tabla_experimento.to_csv(raiz_curso / 'outputs' / '08_01_tool_selection.csv', index=False)

,solicitud,esperada,seleccionada,seleccion_correcta,argumentos_validos,error
0,cuanto es 18 por 4,calculadora,calculadora,True,True,None
1,busca mochila,busqueda_productos,busqueda_productos,True,True,None
2,donde esta mi paquete,busqueda_documentos,busqueda_documentos,True,True,None
3,temperatura de Santiago,clima_mock,busqueda_documentos,False,True,None


Adaptadores LangChain construidos: 4


## 9. Evaluación

La exactitud de selección mide si se eligió el nombre correcto; la validez de argumentos y el éxito de ejecución son métricas distintas. Contaremos cada dimensión y dejaremos en cero las llamadas al modelo en esta ruta local.

In [8]:
metricas = pd.DataFrame([{
    'tool_selection_accuracy': float(tabla_experimento['seleccion_correcta'].mean()),
    'argument_validity_rate': float(tabla_experimento['argumentos_validos'].mean()),
    'tool_execution_success': float(tabla_experimento['argumentos_validos'].mean()),
    'solicitudes': len(tabla_experimento),
    'llamadas_llm': 0,
}])
display(metricas)
metricas.to_csv(raiz_curso / 'outputs' / '08_01_tool_metrics.csv', index=False)

EJECUTAR_LLAMADA_LLM = False
print('Llamadas máximas si se activa la integración:', 1)
if EJECUTAR_LLAMADA_LLM:
    from llm_engineering.models import get_llm
    modelo = get_llm('ollama').bind_tools(crear_herramientas_langchain())
    propuesta = modelo.invoke('Calcula 18 por 4 y usa la calculadora.')
    print('Propuesta del modelo:', propuesta.tool_calls)

,tool_selection_accuracy,argument_validity_rate,tool_execution_success,solicitudes,llamadas_llm
0,0.75,1.0,1.0,4,0


Llamadas máximas si se activa la integración: 1


## 10. Discusión

El proveedor propone una llamada, pero la aplicación conserva autoridad para validarla y ejecutarla. La calculadora usa un AST restringido; las demás herramientas devuelven datos simulados o locales. Function calling no autoriza por sí mismo operaciones sobre cuentas, red o archivos.

## 11. Ejercicios

1. Añade un campo Pydantic obligatorio y mide el rechazo de argumentos incompletos.
2. Amplía el conjunto etiquetado y compara selección por reglas con selección de un modelo local.
3. Diseña permisos diferentes para herramientas de lectura y escritura.
4. Describe cómo impedirías que una llamada propuesta ejecute operaciones no autorizadas.

## 12. Desafío

Añade una herramienta de consulta a una base de datos simulada con argumentos Pydantic, una lista explícita de operaciones permitidas y pruebas contra consultas no autorizadas. Mide selección, validez y tasa de ejecución exitosa.

## 13. Ideas clave

- Texto generado, salida estructurada, llamada de herramienta y ejecución son etapas distintas.
- Valida nombre y argumentos antes de ejecutar.
- La aplicación, no el modelo, decide si una operación está permitida.
- Cuenta y limita cada solicitud externa.